In [1]:
import os
import json
import cv2
import pandas as pd

In [2]:
# Input / Output paths
ROOT_DIR = "Datasets/BdSLW60_Unprocess"
OUTPUT_DIR = "Datasets/BdSLW60_Preprocessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
metadata = []

# Iterate over folders (e.g., W1-W2, W3-W4)
for folder in os.listdir(ROOT_DIR):
    folder_path = os.path.join(ROOT_DIR, folder)
    json_path = os.path.join(folder_path, "output1.json")

    if not os.path.isfile(json_path):
        continue

    print(f"Processing folder: {folder}")

    with open(json_path, "r") as f:
        data = json.load(f)
            # Each JSON file contains multiple words (e.g., "W1", "W2")
    for word, users in data.items():
        word_output_dir = os.path.join(OUTPUT_DIR, word)
        os.makedirs(word_output_dir, exist_ok=True)

        for user, hand_data in users.items():
            for hand_side, file_info in hand_data.items():
                for file_key, info in file_info.items():
                    file_name = info["FileName"] + ".mp4"
                    video_path = os.path.join(folder_path, file_name)
                    if not os.path.isfile(video_path):
                        print(f"⚠️ Missing video: {video_path}")
                        continue

                    cap = cv2.VideoCapture(video_path)
                    fps = float(info["FrameRate"])
                    trials = info["trials"]

                    for trial_id, seg in trials.items():
                        start = int(seg["starting"])
                        end = int(seg["ending"])

                        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
                        frames = []
                        for frame_no in range(start, end + 1):
                            ret, frame = cap.read()
                            if not ret:
                                break
                            frames.append(frame)

                        if not frames:
                            continue

                        # Save the trimmed clip
                        out_name = f"{user}_{word}_trial{trial_id}.mp4"
                        out_path = os.path.join(word_output_dir, out_name)
                        height, width = frames[0].shape[:2]
                        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                        out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
                        for f in frames:
                            out.write(f)
                        out.release()

                        metadata.append({
                            "Word": word,
                            "User": user,
                            "Trial": trial_id,
                            "Orientation": info["Orientation"],
                            "View": info["View"],
                            "Session": info["Session"],
                            "FrameRate": fps,
                            "FileName": out_name,
                            "SourceFile": file_name
                        })

                    cap.release()

Processing folder: W1-2
Processing folder: W11-12
Processing folder: W111-112
Processing folder: W19-20
Processing folder: W211-212
Processing folder: W213-214
Processing folder: W215-216
Processing folder: W217-218
Processing folder: W219-220
Processing folder: W3-4
Processing folder: W351-352
Processing folder: W353-354
Processing folder: W355-356
Processing folder: W357-358
Processing folder: W359-360
Processing folder: W37-38
Processing folder: W39-40
Processing folder: W41-42
Processing folder: W43-44
Processing folder: W45-46
Processing folder: W47-48
Processing folder: W49-50
Processing folder: W5-6
Processing folder: W7-8
Processing folder: W9-10
Processing folder: W91-92
Processing folder: W93-94
Processing folder: W95-96
Processing folder: W97-98
Processing folder: W99-100


In [7]:
# Save metadata to CSV
metadata_path = os.path.join(OUTPUT_DIR, "metadata.csv")
pd.DataFrame(metadata).to_csv(metadata_path, index=False)
print(f"\n✅ Done! Preprocessed videos saved to {OUTPUT_DIR}")
print(f"Metadata saved to {metadata_path}")


✅ Done! Preprocessed videos saved to Datasets/BdSLW60_Preprocessed
Metadata saved to Datasets/BdSLW60_Preprocessed\metadata.csv
